In [ ]:
import os
import cv2
import numpy as np
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.models import Model
from skimage.feature import hog, daisy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

# Load pre-trained InceptionV3 (without top layer)
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
intermediate_layer_model = Model(inputs=base_model.input, outputs=base_model.get_layer('mixed7').output)

def extract_deep_features(img):
    img = cv2.resize(img, (150, 150))
    img = preprocess_input(np.expand_dims(img.astype('float32'), axis=0))
    features = intermediate_layer_model.predict(img)
    return features.flatten()


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
def extract_hog_daisy(gray):
    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys',
                   transform_sqrt=True, feature_vector=True)
    daisy_feat = daisy(gray, step=8, radius=15, rings=2, histograms=6,
                       orientations=8, visualize=False).flatten()
    min_len = min(len(hog_feat), len(daisy_feat))
    return np.hstack([hog_feat[:min_len], daisy_feat[:min_len]])


In [ ]:
def load_data_and_features(dataset_path):
    X = []
    y = []

    for class_name in os.listdir(dataset_path):
        class_folder = os.path.join(dataset_path, class_name)
        if not os.path.isdir(class_folder): continue

        for file in os.listdir(class_folder):
            if not file.lower().endswith(('.jpg', '.png', '.jpeg')): continue
            image_path = os.path.join(class_folder, file)
            img = cv2.imread(image_path)
            if img is None: continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Combine deep features + HOG + DAISY
            deep_feat = extract_deep_features(img)
            hog_daisy_feat = extract_hog_daisy(gray)
            min_len = min(len(deep_feat), len(hog_daisy_feat))
            hybrid_feat = np.hstack([deep_feat[:min_len], hog_daisy_feat[:min_len]])

            X.append(hybrid_feat)
            y.append(class_name)

    return np.array(X), np.array(y)

In [ ]:
dataset_path = "/content/drive/MyDrive/Minor_Project_image_dataset/Final_Resized"  # update this
X, y = load_data_and_features(dataset_path)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {accuracy * 100:.2f}%")


In [ ]:
import cv2
import numpy as np
from skimage.feature import hog, daisy
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model

# Load intermediate InceptionV3 model (same as training)
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
intermediate_layer_model = Model(inputs=base_model.input, outputs=base_model.get_layer('mixed7').output)

# --- Feature extraction functions ---
def extract_deep_features(img):
    img_resized = cv2.resize(img, (150, 150))
    img_preprocessed = preprocess_input(np.expand_dims(img_resized.astype('float32'), axis=0))
    deep_features = intermediate_layer_model.predict(img_preprocessed, verbose=0)
    return deep_features.flatten()

def extract_hog_daisy(gray_img):
    hog_feat = hog(gray_img, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys',
                   transform_sqrt=True, feature_vector=True)
    daisy_feat = daisy(gray_img, step=8, radius=15, rings=2, histograms=6,
                       orientations=8, visualize=False).flatten()
    min_len = min(len(hog_feat), len(daisy_feat))
    return np.hstack([hog_feat[:min_len], daisy_feat[:min_len]])

# --- Prediction function using in-memory model ---
def predict_single_image(image_path, model, le):
    img = cv2.imread(image_path)
    if img is None:
        print("❌ Could not load image.")
        return

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Extract features
    deep_feat = extract_deep_features(img)
    hog_daisy_feat = extract_hog_daisy(gray)

    # Ensure same length
    min_len = min(len(deep_feat), len(hog_daisy_feat))
    hybrid_feat = np.hstack([deep_feat[:min_len], hog_daisy_feat[:min_len]])

    # Predict
    y_pred = model.predict([hybrid_feat])[0]
    predicted_label = le.inverse_transform([y_pred])[0]
    print(f"🔮 Predicted Class: {predicted_label}")


In [ ]:
# Assuming you already have `model` and `le` defined from training
predict_single_image("/content/drive/MyDrive/Minor_Project_image_dataset/Final_Resized/lung_scc/lungscc1066.jpeg", model, le)


🔮 Predicted Class: lung_scc
